# S3-02 `@hobo/identity` draft POC — HOBO-628

Workload identity for the yard: a host attestor measures the launch, and the identity
service mints **JWT-SVIDs bound to the measured tenant** — never the caller's claims. The
gateway verifies against the issuer's JWKS; renewal is **grace-window-gated key rotation**
that keeps old tokens valid; every denial fails closed, minting neither SVID nor audit
event. RSA keys are generated per run, so cells assert on tokens instead of printing them.

In [1]:
import { strict as assert } from "node:assert";
import { createFixtureHostAttestor, IdentityService } from "./src/index.ts";

const NOW = "2026-06-25T15:00:00.000Z";
const IMAGE_DIGEST = "sha256:aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa";
const SCOPE = { port: "database", resource: "shop.db", verbs: ["write"] };

const attestor = createFixtureHostAttestor({ hostId: "host-firecracker-1" });
const measurement = {
  runId: "run-20260625-identity-628",
  tenant: "tenant-a",
  parentPrincipal: { kind: "user", id: "user:platform-admin" },
  codeVersion: {
    digest: "sha256:bbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbb",
    version: "git:628-identity",
  },
  componentImageDigest: IMAGE_DIGEST,
  hostId: "host-firecracker-1",
  issuedAt: "2026-06-25T14:59:30.000Z",
  expiresAt: "2026-06-25T15:01:00.000Z",
  nonce: "nonce-hobo-628",
  isolationTier: "firecracker-microvm",
};

const service = new IdentityService({
  issuerUrl: "https://identity.yard.example",
  genesisKeyId: "yard-genesis-2026-07",
  ttlSeconds: 900,
  renewalGraceSeconds: 300,
});

async function issue(document) {
  return service.issueSvidFromAttestation({
    attestationDocument: document,
    requestedScope: SCOPE,
    audience: "hobo-gateway",
    agentName: "deployer",
    hostPublicKeys: attestor.publicKeys(),
    knownImageDigests: [IMAGE_DIGEST],
    now: NOW,
  });
}

// The caller claims tenant-evil; issuance binds the MEASURED tenant-a.
const document = attestor.createDocument({
  launchMeasurement: measurement,
  claimed: { tenant: "tenant-evil" },
});
const issued = await issue(document);
assert.equal(issued.status, "issued");
assert.equal(issued.svid.subject, "spiffe://yard/tenant-a/agent/deployer");
console.log(`issued: subject=${issued.svid.subject} event=${issued.events[0].type}`);

const gateway = await service.authenticateAtGateway({ token: issued.token, audience: "hobo-gateway", now: NOW });
assert.equal(gateway.status, "authenticated");
console.log(`gateway: status=${gateway.status} tenant=${gateway.tenant}`);

// Rotation inside the grace window swaps the signing key; the old token still verifies.
const rotateAt = "2026-06-25T15:14:30.000Z";
const rotated = await service.rotateSvid({
  identityId: issued.svid.identityId, now: rotateAt, nextKeyId: "yard-rollover-2026-07",
});
assert.equal(rotated.status, "rotated");
const oldStillValid = await service.authenticateAtGateway({ token: issued.token, audience: "hobo-gateway", now: rotateAt });
console.log(`rotated: event=${rotated.events[0].type} newKid=${rotated.svid.signerKid} oldTokenStillValid=${oldStillValid.status === "authenticated"}`);

issued: subject=spiffe://yard/tenant-a/agent/deployer event=identity.Issued
gateway: status=authenticated tenant=tenant-a
rotated: event=identity.Rotated newKid=yard-rollover-2026-07 oldTokenStillValid=true


## Fail-closed denials

Tampered host signatures, expired launch measurements, premature rotation, and scope
widening are all denied — and denials never mint an SVID or an audit event.

In [2]:
import { strict as assert } from "node:assert";
import { createFixtureHostAttestor, IdentityService } from "./src/index.ts";

const NOW = "2026-06-25T15:00:00.000Z";
const IMAGE_DIGEST = "sha256:aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa";
const SCOPE = { port: "database", resource: "shop.db", verbs: ["write"] };

const attestor = createFixtureHostAttestor({ hostId: "host-firecracker-1" });
const measurement = {
  runId: "run-20260625-identity-628",
  tenant: "tenant-a",
  parentPrincipal: { kind: "user", id: "user:platform-admin" },
  codeVersion: {
    digest: "sha256:bbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbb",
    version: "git:628-identity",
  },
  componentImageDigest: IMAGE_DIGEST,
  hostId: "host-firecracker-1",
  issuedAt: "2026-06-25T14:59:30.000Z",
  expiresAt: "2026-06-25T15:01:00.000Z",
  nonce: "nonce-hobo-628",
  isolationTier: "firecracker-microvm",
};

const service = new IdentityService({
  issuerUrl: "https://identity.yard.example",
  genesisKeyId: "yard-genesis-2026-07",
  ttlSeconds: 900,
  renewalGraceSeconds: 300,
});

async function issue(document) {
  return service.issueSvidFromAttestation({
    attestationDocument: document,
    requestedScope: SCOPE,
    audience: "hobo-gateway",
    agentName: "deployer",
    hostPublicKeys: attestor.publicKeys(),
    knownImageDigests: [IMAGE_DIGEST],
    now: NOW,
  });
}

const document = attestor.createDocument({ launchMeasurement: measurement, claimed: { tenant: "tenant-a" } });
const issued = await issue(document);
const before = service.getAuditEvents().length;

const tampered = await issue({ ...document, hostSignature: { ...document.hostSignature, value: "00".repeat(32) } });
console.log(`tampered attestation denied: ${tampered.reason}`);

const expiredDoc = attestor.createDocument({
  launchMeasurement: { ...measurement, issuedAt: "2026-06-25T14:50:00.000Z", expiresAt: "2026-06-25T14:59:00.000Z" },
  claimed: { tenant: "tenant-a" },
});
const expired = await issue(expiredDoc);
console.log(`expired attestation denied: ${expired.reason}`);

const early = await service.rotateSvid({ identityId: issued.svid.identityId, now: "2026-06-25T15:01:00.000Z" });
console.log(`early rotation denied: ${early.reason}`);

const widened = await service.rotateSvidWithScope({
  identityId: issued.svid.identityId,
  requestedScope: { port: "database", resource: "shop.db", verbs: ["write", "delete"] },
  now: "2026-06-25T15:14:30.000Z",
});
console.log(`scope widening denied: ${widened.reason}`);

assert.equal(service.getAuditEvents().length, before);
console.log(`audit log unchanged by denials: events=${service.getAuditEvents().map((e) => e.type).join(",")}`);
console.log("CONTRACT HELD: identity issues attestation-bound, scope-frozen, rotatable JWT-SVIDs and fails closed");

tampered attestation denied: host-signature-invalid
expired attestation denied: launch-measurement-expired
early rotation denied: rotation-outside-renewal-grace
scope widening denied: scope-widening-requires-live-identity
audit log unchanged by denials: events=identity.Issued
CONTRACT HELD: identity issues attestation-bound, scope-frozen, rotatable JWT-SVIDs and fails closed
